# Titanic Survival Prediction

???? ??? ?? ??, ?? ??, ?? ??? ??? ?? ??? ???? ?? ???????. ?? Decision Tree? Ensemble ?? ??? ???? ???? ??? ??? ???? recall/F1-score?? ?? ?????.

## 1. ??? ??

- train: 1,800 rows
- test: 401 rows
- target: `survived` (`yes` / `no`)
- features: `status`, `age`, `sex`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

sns.set_theme(style="whitegrid")

train = pd.read_csv("data/titanic_train.csv")
test = pd.read_csv("data/titanic_test.csv")

print(train.shape, test.shape)
display(train.head())

## 2. ??? ??

??? ??? ?? ??? ??? ???? ?????. ?? ???? ?? ?? ???? ?? ? ???? ??? ???? recall? ?? ???.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

sns.countplot(data=train, x="survived", ax=axes[0])
axes[0].set_title("Target distribution")

sns.barplot(data=train, x="sex", y=(train["survived"] == "yes").astype(int), ax=axes[1])
axes[1].set_title("Survival rate by sex")
axes[1].set_ylabel("survival rate")

sns.barplot(data=train, x="status", y=(train["survived"] == "yes").astype(int), ax=axes[2])
axes[2].set_title("Survival rate by status")
axes[2].set_ylabel("survival rate")

plt.tight_layout()

## 3. ???

??? ??? One-Hot Encoding?? ????, ??? train/test split?? ? ?? ??? ?????.

In [ ]:
X_train = train.drop(columns="survived")
y_train = train["survived"]
X_test = test.drop(columns="survived")
y_test = test["survived"]

preprocess = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), X_train.columns)],
    remainder="drop"
)

models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42),
    "AdaBoost": AdaBoostClassifier(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

results = []
fitted = {}
for name, model in models.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    report = classification_report(y_test, pred, output_dict=True, zero_division=0)
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "yes_precision": report["yes"]["precision"],
        "yes_recall": report["yes"]["recall"],
        "yes_f1": report["yes"]["f1-score"],
    })
    fitted[name] = pipe

results_df = pd.DataFrame(results).sort_values("yes_f1", ascending=False)
display(results_df)

## 4. ?? ?? ??

???????? ??? ???? ?? ????, ?? ???? ? ? ???? ???? ?? ?????. ???? ???(`yes`) F1-score? ?? ?? ??? ?? ??? ?????.

In [ ]:
best_name = results_df.iloc[0]["model"]
best_model = fitted[best_name]
pred = best_model.predict(X_test)

print(f"Best model: {best_name}")
print(classification_report(y_test, pred, zero_division=0))

ConfusionMatrixDisplay.from_predictions(y_test, pred, cmap="Blues")
plt.title(f"Confusion Matrix - {best_name}")
plt.show()

## 5. ??

- ??? ??? ???? ?? ??? ?? ?? ??? ? ????.
- ????? ????? accuracy? ?? ??? ???? recall/F1-score? ?? ???? ?? ????.
- ?? ?? ??: ??/?? ?/?? ? ? ??? ?? ??, class imbalance ??, ???? ?? ??????? ??.